# Módulo 1: Pipeline 16S con DADA2 y QIIME2

En este notebook vamos a procesar lecturas de 16S rRNA desde archivos FASTQ crudos hasta obtener:
- Una tabla de ASVs
- Asignación taxonómica
- Métricas de diversidad alfa y beta
- Gráficos taxonómicos

## Flujo del pipeline

```
FASTQ crudos
    └─► Importar a QIIME2
            └─► Visualizar calidad
                    └─► Remover primers (cutadapt)
                            └─► Denoising (DADA2)
                                    └─► Filogenia
                                            └─► Diversidad alfa y beta
                                                    └─► Taxonomía
```

## Requisitos

Asegúrate de tener el entorno activo antes de abrir este notebook:

```bash
conda activate qiime2-amplicon-2025.4
jupyter notebook
```

## 0. Configuración de rutas

Definimos todas las rutas en un solo lugar para que sea fácil adaptar el notebook a un dataset distinto.

In [ ]:
import os

# --- Rutas ---
DATA_DIR      = "../data/raw_reads"
RESULTS_DIR   = "../results"
METADATA      = "../data/metadata.tsv"
MANIFEST      = "../data/manifest.tsv"
CLASSIFIER    = "../data/taxonomy_db/silva-138-99-515-806.qza"  # descarga instrucciones al final

# --- Primers 16S (región V4) ---
# ⚠️  Verifica que estos primers correspondan a tu experimento
FWD_PRIMER = "GTGYCAGCMGCCGCGGTAA"   # 515F
REV_PRIMER = "GGACTACNVGGGTWTCTAAT"  # 806R

# Crear carpeta de resultados
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Rutas configuradas correctamente")

## 1. Manifest y metadata

QIIME2 necesita un **manifest** para saber dónde están los archivos FASTQ de cada muestra, y un archivo de **metadata** con información sobre las muestras.

### Manifest
Tiene tres columnas: `sample-id`, `forward-absolute-filepath`, `reverse-absolute-filepath`.

### Metadata
Tiene una fila por muestra con variables de interés (tratamiento, timepoint, etc.).

Vamos a generarlos automáticamente a partir de los archivos en `raw_reads/`.

In [ ]:
import pandas as pd
import glob

# --- Generar manifest ---
raw_reads_abs = os.path.abspath(DATA_DIR)
r1_files = sorted(glob.glob(os.path.join(raw_reads_abs, "*_R1.fastq.gz")))

rows = []
for r1 in r1_files:
    sample_id = os.path.basename(r1).replace("_R1.fastq.gz", "")
    r2 = r1.replace("_R1.fastq.gz", "_R2.fastq.gz")
    rows.append({"sample-id": sample_id,
                 "forward-absolute-filepath": r1,
                 "reverse-absolute-filepath": r2})

manifest = pd.DataFrame(rows)
manifest.to_csv(MANIFEST, sep="\t", index=False)
print("Manifest generado:")
manifest

In [ ]:
# --- Visualizar metadata ---
metadata = pd.read_csv(METADATA, sep="\t")
print("Metadata:")
metadata

## 2. Importar secuencias a QIIME2

QIIME2 trabaja con archivos `.qza` (QIIME2 Artifact). El primer paso es importar los FASTQ usando el manifest que generamos.

- **Tipo:** `SampleData[PairedEndSequencesWithQuality]`
- **Formato:** `PairedEndFastqManifestPhred33V2`

In [ ]:
!qiime tools import \
    --type 'SampleData[PairedEndSequencesWithQuality]' \
    --input-path {MANIFEST} \
    --output-path {RESULTS_DIR}/sequences.qza \
    --input-format PairedEndFastqManifestPhred33V2

print("✓ Secuencias importadas:", RESULTS_DIR + "/sequences.qza")

## 3. Visualizar calidad de las lecturas

Antes de hacer el denoising necesitamos ver la calidad de las lecturas para decidir los parámetros de truncado.

El archivo `.qzv` se puede visualizar en [https://view.qiime2.org](https://view.qiime2.org).

In [ ]:
!qiime demux summarize \
    --i-data {RESULTS_DIR}/sequences.qza \
    --o-visualization {RESULTS_DIR}/sequences_summary.qzv

print("✓ Resumen de calidad generado")
print("  → Visualiza en https://view.qiime2.org subiendo:", RESULTS_DIR + "/sequences_summary.qzv")

### ¿Qué buscar en el gráfico de calidad?

- El eje X muestra la posición en la lectura (pb)
- El eje Y muestra el Phred quality score (Q)
- **Q ≥ 30** = 99.9% de precisión (buena calidad)
- **Q < 20** = zona de baja calidad, considerar truncar

Usa este gráfico para definir `--p-trunc-len-f` y `--p-trunc-len-r` en el paso de denoising.

## 4. Remover primers con Cutadapt

Los primers deben removerse antes del denoising. Si no se eliminan, DADA2 los puede interpretar como variación biológica real y generar ASVs erróneos.

⚠️ **Reemplaza los primers si usaste un par distinto al V4 (515F/806R).**

In [ ]:
!qiime cutadapt trim-paired \
    --i-demultiplexed-sequences {RESULTS_DIR}/sequences.qza \
    --p-front-f {FWD_PRIMER} \
    --p-front-r {REV_PRIMER} \
    --p-discard-untrimmed \
    --p-cores 4 \
    --o-trimmed-sequences {RESULTS_DIR}/sequences_trimmed.qza \
    --verbose 2>&1 | tail -20

print("✓ Primers removidos")

In [ ]:
# Visualizar calidad post-trimming
!qiime demux summarize \
    --i-data {RESULTS_DIR}/sequences_trimmed.qza \
    --o-visualization {RESULTS_DIR}/sequences_trimmed_summary.qzv

print("✓ Visualiza la calidad post-trimming en https://view.qiime2.org")

## 5. Denoising con DADA2

DADA2 hace cuatro cosas en un solo paso:
1. **Filtra** lecturas de baja calidad
2. **Aprende** el modelo de error de la corrida
3. **Deniosa** y corrige errores de secuenciación
4. **Elimina** quimeras

El resultado son **ASVs** (Amplicon Sequence Variants) — secuencias exactas, más precisas que OTUs.

### ASVs vs OTUs
| | OTUs | ASVs |
|---|---|---|
| Similitud | 97% | 100% (secuencia exacta) |
| Resolución | Género/especie | Sub-especie |
| Reproducibilidad | Depende del umbral | Alta |
| Método | Clustering | Denoising |

### Parámetros clave
- `--p-trunc-len-f` / `--p-trunc-len-r`: truncar donde la calidad cae. Ajusta según el gráfico del paso 3.
- Las lecturas R1 y R2 deben solapar al menos 20pb después del truncado.

In [ ]:
# ⚠️ Ajusta trunc-len-f y trunc-len-r según el gráfico de calidad del paso 3
TRUNC_F = 230
TRUNC_R = 200

!qiime dada2 denoise-paired \
    --i-demultiplexed-seqs {RESULTS_DIR}/sequences_trimmed.qza \
    --p-trunc-len-f {TRUNC_F} \
    --p-trunc-len-r {TRUNC_R} \
    --p-n-threads 4 \
    --o-table {RESULTS_DIR}/asv_table.qza \
    --o-representative-sequences {RESULTS_DIR}/rep_seqs.qza \
    --o-denoising-stats {RESULTS_DIR}/denoising_stats.qza

print("✓ Denoising completado")

## 6. Estadísticas de denoising

Verificamos cuántas lecturas pasaron cada paso del filtrado. Si muchas lecturas se pierden en algún paso, puede indicar un problema con los parámetros.

In [ ]:
!qiime metadata tabulate \
    --m-input-file {RESULTS_DIR}/denoising_stats.qza \
    --o-visualization {RESULTS_DIR}/denoising_stats.qzv

print("✓ Estadísticas de denoising generadas")
print("  → Visualiza en https://view.qiime2.org")

### ¿Qué esperar?

| Columna | Descripción | Valor esperado |
|---|---|---|
| `input` | Lecturas totales | 100% |
| `filtered` | Post filtro de calidad | > 80% |
| `denoised` | Post corrección de errores | ~ filtered |
| `merged` | R1+R2 solapados | > 70% |
| `non-chimeric` | Post eliminación de quimeras | > 70% del input |

## 7. Tabla ASV y secuencias representativas

Visualizamos la tabla ASV para ver cuántos ASVs y lecturas tenemos por muestra.

In [ ]:
!qiime feature-table summarize \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --m-sample-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/asv_table_summary.qzv

!qiime feature-table tabulate-seqs \
    --i-data {RESULTS_DIR}/rep_seqs.qza \
    --o-visualization {RESULTS_DIR}/rep_seqs.qzv

print("✓ Tabla ASV y secuencias representativas generadas")

## 8. Filogenia

Construimos un árbol filogenético a partir de las secuencias representativas. Es necesario para calcular métricas de diversidad que consideran distancias evolutivas (UniFrac).

El comando `align-to-tree-mafft-fasttree` hace en un paso:
1. Alineamiento múltiple con MAFFT
2. Enmascarar posiciones poco informativas
3. Construir árbol con FastTree
4. Enraizar el árbol

In [ ]:
!qiime phylogeny align-to-tree-mafft-fasttree \
    --i-sequences {RESULTS_DIR}/rep_seqs.qza \
    --o-alignment {RESULTS_DIR}/aligned_rep_seqs.qza \
    --o-masked-alignment {RESULTS_DIR}/masked_aligned_rep_seqs.qza \
    --o-tree {RESULTS_DIR}/unrooted_tree.qza \
    --o-rooted-tree {RESULTS_DIR}/rooted_tree.qza

print("✓ Árbol filogenético construido")

## 9. Diversidad alfa y beta

### Diversidad alfa
Mide la diversidad **dentro** de una muestra.
- **Shannon:** considera riqueza y abundancia relativa
- **Observed features:** número de ASVs únicos
- **Faith's PD:** diversidad filogenética

### Diversidad beta
Mide la diversidad **entre** muestras.
- **Bray-Curtis:** basada en abundancias, sin filogenia
- **UniFrac no ponderado:** basada en presencia/ausencia + filogenia
- **UniFrac ponderado:** basada en abundancias + filogenia

### Rarefacción
Antes de calcular diversidad hay que normalizar por profundidad de secuenciación (rarefacción). El parámetro `--p-sampling-depth` define la profundidad mínima — muestras con menos lecturas serán excluidas.

Revisa el resumen de la tabla ASV para elegir un valor que retenga la mayoría de las muestras.

In [ ]:
# ⚠️ Ajusta según el mínimo de lecturas por muestra visto en el paso 7
SAMPLING_DEPTH = 5000

!qiime diversity core-metrics-phylogenetic \
    --i-phylogeny {RESULTS_DIR}/rooted_tree.qza \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --p-sampling-depth {SAMPLING_DEPTH} \
    --m-metadata-file {METADATA} \
    --output-dir {RESULTS_DIR}/diversity

print("✓ Métricas de diversidad calculadas en:", RESULTS_DIR + "/diversity/")

In [ ]:
# Curvas de rarefacción — ver a qué profundidad se estabiliza la diversidad
!qiime diversity alpha-rarefaction \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --i-phylogeny {RESULTS_DIR}/rooted_tree.qza \
    --p-max-depth {SAMPLING_DEPTH} \
    --m-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/alpha_rarefaction.qzv

print("✓ Curvas de rarefacción generadas")

In [ ]:
# Test estadístico: diferencias en diversidad alfa entre grupos
!qiime diversity alpha-group-significance \
    --i-alpha-diversity {RESULTS_DIR}/diversity/shannon_vector.qza \
    --m-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/diversity/shannon_significance.qzv

print("✓ Test de significancia de diversidad alfa (Shannon) generado")

In [ ]:
# Test estadístico: diferencias en composición entre grupos (PERMANOVA)
!qiime diversity beta-group-significance \
    --i-distance-matrix {RESULTS_DIR}/diversity/bray_curtis_distance_matrix.qza \
    --m-metadata-file {METADATA} \
    --m-metadata-column Treatment \
    --o-visualization {RESULTS_DIR}/diversity/bray_curtis_significance.qzv

print("✓ Test PERMANOVA (Bray-Curtis) generado")

### PCoA — Ordenación de muestras

El PCoA (Principal Coordinates Analysis) muestra la similitud entre muestras en un espacio reducido. Muestras cercanas tienen composición microbiana similar.

Los archivos `*_emperor.qzv` en la carpeta de diversidad contienen el PCoA interactivo.

In [ ]:
import os
pcoa_files = [f for f in os.listdir(RESULTS_DIR + "/diversity") if "emperor" in f]
print("Archivos PCoA disponibles:")
for f in pcoa_files:
    print(" →", f)
print("\nVisualiza en https://view.qiime2.org")

## 10. Asignación taxonómica

Asignamos taxonomía a cada ASV usando un clasificador entrenado con la base de datos Silva.

### Descargar el clasificador Silva (una sola vez)

```bash
wget -P ../data/taxonomy_db/ \
    https://data.qiime2.org/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-classifier.qza
```

> El archivo pesa ~1.7GB. Descárgalo con anticipación.

In [ ]:
CLASSIFIER = "../data/taxonomy_db/silva-138-99-nb-classifier.qza"

!qiime feature-classifier classify-sklearn \
    --i-classifier {CLASSIFIER} \
    --i-reads {RESULTS_DIR}/rep_seqs.qza \
    --p-n-jobs -1 \
    --o-classification {RESULTS_DIR}/taxonomy.qza

print("✓ Taxonomía asignada")

In [ ]:
# Visualizar tabla de taxonomía
!qiime metadata tabulate \
    --m-input-file {RESULTS_DIR}/taxonomy.qza \
    --o-visualization {RESULTS_DIR}/taxonomy.qzv

print("✓ Tabla de taxonomía generada")

## 11. Gráficos de abundancia relativa (barplots)

Los barplots muestran la composición taxonómica de cada muestra. Puedes cambiar el nivel taxonómico (Filo, Clase, Orden, Familia, Género) en el visualizador interactivo.

In [ ]:
!qiime taxa barplot \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --i-taxonomy {RESULTS_DIR}/taxonomy.qza \
    --m-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/taxa_barplot.qzv

print("✓ Barplot taxonómico generado")
print("  → Visualiza en https://view.qiime2.org")

## 12. Resumen de archivos generados

Al terminar el pipeline deberías tener los siguientes archivos en `results/`:

In [ ]:
import os

print(f"Contenido de {RESULTS_DIR}:\n")
for root, dirs, files in os.walk(RESULTS_DIR):
    level = root.replace(RESULTS_DIR, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    if level > 0:
        print(f"{indent}{folder}/")
    for f in sorted(files):
        print(f"{indent}  {f}")

## Referencia rápida: archivos `.qza` vs `.qzv`

| Extensión | Tipo | Uso |
|---|---|---|
| `.qza` | Artifact | Datos procesados, input para próximos pasos |
| `.qzv` | Visualization | Solo para visualizar en view.qiime2.org |

Ambos son archivos ZIP — puedes renombrarlos a `.zip` y abrirlos para inspeccionar su contenido.